# Weighted Evaluation (MEPS LONGWT)

This notebook reports **survey-weighted model performance** using MEPS **longitudinal person weights (`LONGWT`)**.  
The goal is to check whether conclusions from the unweighted test evaluation remain consistent when performance is computed in a way that is more representative of the U.S. population.

We focus on three Year-2 classification tasks and reuse the best-performing baseline model family for each task:
- **HIGHCOST_Y2:** Random Forest
- **ANY_ED_Y2:** XGBoost
- **ANY_IP_Y2:** Random Forest

**Key idea:**  
Unweighted metrics treat each sampled person equally. Weighted metrics use `LONGWT` so each observation contributes proportionally to the population it represents.


## 0) Setup and imports

**Goal:** Import packages and model classes needed for weighted evaluation.

We use:
- `RandomForestClassifier` and `XGBClassifier` for the selected tasks
- `roc_auc_score` and `average_precision_score` with `sample_weight=LONGWT` for weighted ROC-AUC and PR-AUC


In [62]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import roc_auc_score, average_precision_score


## 1) Load the feature dataset (`df_feat`)

**Goal:** Load the modeling-ready feature table created in earlier notebooks (`df_feat.parquet`).

Feature engineering has already been completed (see Notebook 02).  
Here we only **fit models and compute weighted metrics** on a consistent train/val/test split.


In [ ]:
PROJECT_ROOT = Path("..").resolve()  
DATA_DIR = PROJECT_ROOT / "data"

df_feat = pd.read_parquet(DATA_DIR / "df_feat.parquet")
df_feat.shape


(7812, 141)

## 2) Import shared utilities from `src.models`

**Goal:** Reuse the same splitting and preprocessing logic as previous notebooks to ensure comparability.

- `split_train_val_test`: consistent train/validation/test split (with optional stratification)
- `make_preprocess`: `ColumnTransformer` preprocessing (imputation + one-hot encoding)

**Note:** For tree-based models, scaling is not required, so we set `scale_numeric=False`.


In [61]:
import sys
sys.path.append(str(PROJECT_ROOT))

from src.models import split_train_val_test
from src.models import make_preprocess  

## 3) Survey weights and design variables

**Goal:** Confirm the availability of MEPS survey weight columns.

We use:
- `LONGWT` as the **person-level longitudinal weight** for weighted performance estimates.

We also check `VARSTR` and `VARPSU` (design variables), but **this notebook reports weighted point estimates only** (AUC/PR-AUC/precision/recall).  
Complex-survey variance estimation (SEs/CI) would require survey-design methods and is not implemented here.


In [48]:
W_COL = "LONGWT"
STR_COL = "VARSTR"
PSU_COL = "VARPSU"

df_feat[[W_COL, STR_COL, PSU_COL]].isna().sum()


LONGWT    0
VARSTR    0
VARPSU    0
dtype: int64

## 4) Define the feature set (categorical + numeric)

**Goal:** Use a consistent, interpretable feature set aligned with the thesis baseline.

- `cat_cols`: categorical predictors to be one-hot encoded (race/ethnicity, region, education, poverty category, family size group, insurance type)
- `num_cols`: engineered numeric predictors (demographics, SES, employment, health status, chronic conditions, baseline utilization/cost)

We then combine them into:
- `FEATURES = num_cols + cat_cols`


In [49]:
cat_cols = [
    "RACE_ETH",
    "REGIONY1_CAT",
    "EDU_GROUP",
    "POVCATY1_CAT",
    "FAMSIZE_Y1_GRP",
    "INS_TYPE_Y1",
]

In [50]:
#Numeric (use engineered columns, not raw)

num_cols = [
    # demographics / SES
    "AGE",
    "SEX_BIN",
    "LOG_FAMINCY1",
    "FAMSIZE_Y1",

   

    # employment
    "WORKED_Y1",
    "ANY_UNEMP_COMP_Y1",
    "LOG_UNEMP_COMP_Y1",
    "EMP_INFO_R12",
    "EMP_ATTACHED_ANY_R12_FILL0",  # model-friendly version

    # health status baseline
    "RTHLTH1_FAIRPOOR",
    "MNHLTH1_FAIRPOOR",

    # chronic conditions baseline
    "HIBPDXY1_BIN",
    "CHDDXY1_BIN",
    "STRKDXY1_BIN",
    "CHOLDXY1_BIN",
    "ASTHDXY1_BIN",
    "DIABDXY1_M18_BIN",
    # "MULTIMORBIDITY_Y1",   # optional (can remove if you keep all *_BIN)

    # baseline utilisation/cost
    "LOG_TOTEXPY1",
    "ANY_ED_Y1",
    "ANY_IP_Y1",
]

In [51]:
FEATURES = num_cols + cat_cols


## 5) Train/validation/test split with aligned weights

**Goal:** Create the same split as before, while keeping the **test weights aligned** with the test rows.

We:
1) drop missing target/weight,
2) split X/y into train/val/test ,
3) retrieve `w_test` using the test indices.

This ensures `y_test`, `proba_test`, and `w_test` are perfectly aligned for weighted scoring.


In [56]:
def build_split_with_weights(df, target, feature_cols, *, random_state=42, stratify=False):
    tmp = df[feature_cols + [target, W_COL]].copy()
    tmp = tmp[tmp[target].notna() & tmp[W_COL].notna()]  # only drop missing target/weight

    X = tmp[feature_cols].copy()
    y = tmp[target].copy()

    X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
        X, y, random_state=random_state, stratify=stratify
    )

    w_test = tmp.loc[X_test.index, W_COL].astype(float).to_numpy()
    return X_train, X_val, X_test, y_train, y_val, y_test, w_test


## 6) Weighted metrics (how we evaluate)

**Goal:** Compute population-weighted performance on the test set.

We report:
- **Weighted ROC-AUC** using `roc_auc_score(..., sample_weight=LONGWT)`
- **Weighted PR-AUC** using `average_precision_score(..., sample_weight=LONGWT)`

For a fixed operating threshold `t`, we also compute:
- **Weighted precision** and **weighted recall**

This lets us compare both:
- ranking quality (AUC/PR-AUC), and
- operational trade-offs at a chosen threshold (precision/recall)


In [57]:
def weighted_clf_metrics(y_true, proba, w, threshold):
    y_true = np.asarray(y_true).astype(int)
    proba = np.asarray(proba).astype(float)
    w = np.asarray(w).astype(float)

    auc_w = roc_auc_score(y_true, proba, sample_weight=w)
    pr_w = average_precision_score(y_true, proba, sample_weight=w)

    pred = (proba >= threshold).astype(int)
    tp = w[(y_true==1) & (pred==1)].sum()
    fp = w[(y_true==0) & (pred==1)].sum()
    fn = w[(y_true==1) & (pred==0)].sum()

    precision_w = tp / (tp + fp + 1e-12)
    recall_w = tp / (tp + fn + 1e-12)

    return {"AUC_w": float(auc_w), "PR_AUC_w": float(pr_w),
            "precision_w": float(precision_w), "recall_w": float(recall_w)}


## 7) Weighted evaluation: HIGHCOST_Y2 (Random Forest)

**Goal:** Re-fit the selected model and compute **weighted** test metrics.

Model choice:
- Random Forest is a strong baseline for high-cost prediction and performs well without requiring feature scaling.

We evaluate weighted AUC/PR-AUC and weighted precision/recall at a fixed threshold (`t_hc`), chosen from prior threshold analysis.


In [58]:
RANDOM_SEED = 42

# preprocessing
pre_tree = make_preprocess(num_cols,cat_cols, scale_numeric=False)

# ---- HIGHCOST RF ----
X_tr, X_va, X_te, y_tr, y_va, y_te, w_te = build_split_with_weights(
    df_feat, "HIGHCOST_Y2", FEATURES, random_state=RANDOM_SEED, stratify=True
)
rf_hc = RandomForestClassifier(
    n_estimators=800, max_depth=16, min_samples_leaf=3,
    random_state=RANDOM_SEED, n_jobs=-1, class_weight="balanced_subsample"
)
pipe_hc = Pipeline([("preprocess", pre_tree), ("model", rf_hc)])
pipe_hc.fit(X_tr, y_tr)
proba_hc = pipe_hc.predict_proba(X_te)[:, 1]

t_hc = 0.55
hc_w = weighted_clf_metrics(y_te.values, proba_hc, w_te, threshold=t_hc)
hc_w


{'AUC_w': 0.8452712147296142,
 'PR_AUC_w': 0.4188561576848545,
 'precision_w': 0.5399133056652324,
 'recall_w': 0.3245361254638}

## 8) Weighted evaluation: ANY_ED_Y2 (XGBoost)

**Goal:** Re-fit the selected ED model and compute **weighted** test metrics.

Model choice:
- XGBoost provided better ranking (PR-AUC) than Random Forest for ED risk in earlier comparisons.

We evaluate weighted AUC/PR-AUC and weighted precision/recall at a fixed threshold (`t_ed`) consistent with the chosen operating point.


In [59]:
# ---- ANY_ED XGB ----
X_tr, X_va, X_te, y_tr, y_va, y_te, w_te = build_split_with_weights(
    df_feat, "ANY_ED_Y2", FEATURES, random_state=RANDOM_SEED, stratify=True
)

xgb_ed = XGBClassifier(
    n_estimators=800, max_depth=3, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.9,
    random_state=RANDOM_SEED, n_jobs=-1, tree_method="hist",
    eval_metric="logloss"
)
pipe_ed = Pipeline([("preprocess", pre_tree), ("model", xgb_ed)])
pipe_ed.fit(X_tr, y_tr)
proba_ed = pipe_ed.predict_proba(X_te)[:, 1]

t_ed = 0.20
ed_w = weighted_clf_metrics(y_te.values, proba_ed, w_te, threshold=t_ed)
ed_w


{'AUC_w': 0.7202597849413098,
 'PR_AUC_w': 0.3514076410116459,
 'precision_w': 0.2924844814749709,
 'recall_w': 0.4330723017721842}

## 9) Weighted evaluation: ANY_IP_Y2 (Random Forest)

**Goal:** Re-fit the selected inpatient model and compute **weighted** test metrics.

Model choice:
- Random Forest performed comparably or better than XGBoost for inpatient prediction in our benchmarks.

Because inpatient events are rarer, we expect stronger class-imbalance effects. Weighted metrics help assess whether performance conclusions hold at the population level.


In [60]:
# ---- ANY_IP RF ----
X_tr, X_va, X_te, y_tr, y_va, y_te, w_te = build_split_with_weights(
    df_feat, "ANY_IP_Y2", FEATURES, random_state=RANDOM_SEED, stratify=True
)

rf_ip = RandomForestClassifier(
    n_estimators=800, max_depth=16, min_samples_leaf=3,
    random_state=RANDOM_SEED, n_jobs=-1, class_weight="balanced_subsample"
)
pipe_ip = Pipeline([("preprocess", pre_tree), ("model", rf_ip)])
pipe_ip.fit(X_tr, y_tr)
proba_ip = pipe_ip.predict_proba(X_te)[:, 1]

t_ip = 0.40
ip_w = weighted_clf_metrics(y_te.values, proba_ip, w_te, threshold=t_ip)
ip_w


{'AUC_w': 0.7593959584403898,
 'PR_AUC_w': 0.18965983656913243,
 'precision_w': 0.274283605648523,
 'recall_w': 0.2826755702834657}

## Weighted vs. unweighted evaluation (LONGWT)

We compared standard **unweighted** test metrics to **survey-weighted** metrics using MEPS longitudinal weights (`LONGWT`). Weighted results reflect population-representative performance and can shift operating-point metrics (precision/recall) even when ranking metrics (AUC/PR-AUC) are similar.

### HIGHCOST_Y2 (Random Forest), threshold t = 0.55
- **AUC:** 0.868 → **0.845** (weighted slightly lower)
- **PR-AUC:** 0.459 → **0.419** (weighted lower)
- **Precision:** 0.487 → **0.540** (weighted higher)
- **Recall:** 0.596 → **0.325** (weighted lower)

**Interpretation:** Ranking performance decreases modestly under weighting, and the fixed threshold corresponds to a more conservative operating point in the weighted population (higher precision but substantially lower recall).

### ANY_ED_Y2 (XGBoost), threshold t = 0.20
- **AUC:** 0.728 → **0.720** (very similar)
- **PR-AUC:** 0.350 → **0.351** (essentially unchanged)
- **Precision:** 0.313 → **0.292** (slightly lower)
- **Recall:** 0.439 → **0.433** (very similar)

**Interpretation:** ED results are highly consistent between weighted and unweighted evaluation, suggesting robust population-level generalization for this task.

### ANY_IP_Y2 (Random Forest), threshold t = 0.40
- **AUC:** 0.755 → **0.759** (slightly higher)
- **PR-AUC:** 0.225 → **0.190** (weighted lower)
- **Precision:** 0.234 → **0.274** (weighted higher)
- **Recall:** 0.363 → **0.283** (weighted lower)

**Interpretation:** Discrimination (AUC) remains similar, but weighted PR-AUC decreases and the fixed threshold shifts toward higher precision and lower recall.

### Overall takeaway
Survey weighting does not materially change the main ranking conclusions (AUC/PR-AUC are broadly similar), but it can **re-balance precision vs recall at a fixed threshold**, because the weighted population composition differs from the raw sample. For deployment-style decisions, threshold calibration may need to be revisited under weighted evaluation.

Overall conclusions were unchanged: high-cost prediction remains the strongest task, ED is moderate, and inpatient admission is the most challenging due to rarity.